In [1]:
import sys; sys.path.append("../..")
from donnees import charger_compteurs, parties, seuil, ecrire_predictions, COLONNES_E

In [2]:
df = charger_compteurs()
df.head()

,block_id,label,split,E1,E2,E3,E4,E5,E6,E7,...,E26,E27,E28,E29,E30,n_warn,n_error,length,n_unique,duration_s
0,blk_-1608999687919862906,0,train,0,0,203,0,10,7,0,...,10,0,0,0,0,0,0,269,12,3802
1,blk_7503483334202473044,0,train,0,2,1,0,3,0,0,...,3,0,0,0,0,0,0,22,9,3802
2,blk_-3544583377289625738,1,train,0,0,203,0,3,0,0,...,3,0,0,0,0,1,0,223,9,3797
3,blk_-9073992586687739851,0,train,0,3,0,0,3,0,0,...,3,0,0,0,0,0,0,22,8,50448
4,blk_7854771516489510256,0,train,0,3,1,15,3,0,0,...,3,0,0,0,0,15,0,38,10,50583


In [3]:
print(df.shape)                       # attendu : (575061, nombre_de_colonnes)
print(df["split"].value_counts())     # train 402542, val 86259, test 86260
print(df.groupby("split")["label"].mean())   # part d'anomalies : 3.26%, 3.02%, 1.29%

(575061, 38)
split
train    402542
test      86260
val       86259
Name: count, dtype: int64
split
test     0.012938
train    0.032580
val      0.030223
Name: label, dtype: float64


In [ ]:
import numpy as np
import time
from sklearn.ensemble import IsolationForest
from sklearn.metrics import f1_score

p = parties(df)

X_train      = p["train_normal"][COLONNES_E].values
X_val        = p["val"][COLONNES_E].values
y_val        = p["val"]["label"].values
X_val_normal = p["val_normal"][COLONNES_E].values
X_test       = p["test"][COLONNES_E].values

In [5]:
import time
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler

In [6]:
# Sous-échantillonner l'ensemble d'entraînement pour accélérer l'entraînement du modèle.
rng = np.random.default_rng(42)
idx = rng.choice(len(X_train), size=20000, replace=False)
X_sub = X_train[idx]
print(X_sub.shape)    # (20000, 30)


(20000, 30)


In [7]:
# Normaliser les données pour le OneClassSVM.
scaler = StandardScaler().fit(X_sub)

def score(model, X):
    return -model.score_samples(scaler.transform(X))

In [8]:
# Tester différentes combinaisons de paramètres pour le OneClassSVM.
resultats = {}
for nu in [0.001, 0.01, 0.05]:
    for gamma in ["scale", 0.01, 0.1]:
        model = OneClassSVM(nu=nu, gamma=gamma).fit(scaler.transform(X_sub))
        s = seuil(score(model, X_val_normal))
        f1 = f1_score(y_val, score(model, X_val) > s)
        resultats[(nu, gamma)] = f1
        print(nu, gamma, round(f1, 4))

meilleur = max(resultats, key=resultats.get)
print("Meilleur :", meilleur)

0.001 scale 0.9943
0.001 0.01 0.9985
0.001 0.1 0.9958
0.01 scale 0.9926
0.01 0.01 0.9558
0.01 0.1 0.9926
0.05 scale 0.9558
0.05 0.01 0.9558
0.05 0.1 0.9558
Meilleur : (0.001, 0.01)


In [9]:
t0 = time.time()
model = OneClassSVM(nu=meilleur[0], gamma=meilleur[1]).fit(scaler.transform(X_sub))
temps_train = (time.time() - t0) / 60
print("Entraînement (min) :", round(temps_train, 2))

s = seuil(score(model, X_val_normal))

t0 = time.time()
scores_test = score(model, X_test)
temps_pred = time.time() - t0
print("Prédiction (s) :", round(temps_pred, 1))

ecrire_predictions(p["test"], scores_test, s, "../../results/one_class_svm_test.csv")

Entraînement (min) : 0.0
Prédiction (s) : 0.2


,block_id,score,pred
0,blk_3458545012440520721,-2.718251,0
1,blk_-1160437354072569825,-2.718251,0
2,blk_-8961379332018832424,-2.718251,0
3,blk_-7458466477853961957,-2.718251,0
4,blk_3070076663445833576,-2.718251,0
...,...,...,...
86255,blk_1019720114020043203,-2.805893,0
86256,blk_-2683116845478050414,-2.805893,0
86257,blk_5595059397348477632,-2.805893,0
86258,blk_1513937873877967730,-2.805893,0


In [10]:
from sklearn.metrics import precision_score, recall_score, confusion_matrix

model = OneClassSVM(nu=0.001, gamma=0.01).fit(scaler.transform(X_sub))
s = seuil(score(model, X_val_normal))
pred_val = score(model, X_val) > s

print("Seuil :", s)
print("Alertes sur val :", pred_val.sum(), "sur", len(pred_val))
print("Anomalies réelles sur val :", y_val.sum())
print("Précision :", round(precision_score(y_val, pred_val), 4))
print("Rappel    :", round(recall_score(y_val, pred_val), 4))
print(confusion_matrix(y_val, pred_val))
print("Scores distincts sur val :", len(np.unique(score(model, X_val))))

# Comparaison avec la base de Mouad
base = p["val"]["n_warn"].values > 0
print("F1 base (n_warn>0) sur val :", round(f1_score(y_val, base), 4))

Seuil : -2.718250955596768
Alertes sur val : 2613 sur 86259
Anomalies réelles sur val : 2607
Précision : 0.9973
Rappel    : 0.9996
[[83645     7]
 [    1  2606]]
Scores distincts sur val : 76
F1 base (n_warn>0) sur val : 0.718


In [12]:
import pandas as pd
pred_test = pd.read_csv("../../results/one_class_svm_test.csv")
print(pred_test["pred"].sum(), "alertes sur", len(pred_test))

1197 alertes sur 86260
